# Week 10 Tutorial --- Dispatching 1

In [ ]:
import enn555.wind as wind
import pandas as pd
import gurobipy as gp
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from os import listdir
import matplotlib.pyplot as plt
import numpy as np
from enn555.paths import data_dir
from enn555.nem_prices import download_price_and_demand
from pvlib.iotools import get_nasa_power
dd = data_dir()
data_path = dd/"dispatching_tutorials"

## Get 2024 data from NASA Power file

In [ ]:
lat,lon = -38.4, 147.4
hub_height = 100
now = datetime(2024,10,5,0,0,tzinfo=timezone(timedelta(hours=10)))
end = now + timedelta(hours=5*24) # 5 day optimization horizon
re_download = True

Note that I have to download the whole year to simulate in SAM and then focus on the horizon of interest

In [ ]:
if re_download:
    
    # Adjust year to account for shift due to moving from UTC to UTC+10
    start_year,end_year = str(now.year-1)+"-12-31",str(now.year)+"-12-31"

    df,meta = get_nasa_power(latitude=lat,longitude=lon,start=start_year,end=end_year,
                            parameters=["T2M","PS","WS10M","WD10M","WS50M","WD50M"],
                            map_variables=False)

    start_year, end_year = pd.Timestamp(start_year+"T14:00", tz="UTC"),pd.Timestamp(end_year+"T14:00", tz="UTC")
    mask = (df.index >= start_year) & (df.index < end_year)   # [start_year, end_year)
    df = df.loc[mask]

    gippsland = wind.merra_wind_speed_data()
    gippsland.import_dataframe(df)
    gippsland.latitude = lat
    gippsland.longitude = lon
    gippsland.utc_to_lst(10) # we only really need to do this if we are aligning with the load.
    print(f"Downloaded from {gippsland.data.Timestamp[0]} to {gippsland.data.Timestamp.iloc[-1]}")

    gippsland.add_speed_at_height(hub_height,alpha=0.14)
    gippsland.export_to_sam_csv(utc_offset=10,file_name=data_path/"sam_export_2024_ALL_YEAR")

    gippsland.data.set_index('Timestamp',inplace=True)


    # prices
    df_prices = download_price_and_demand(region="VIC1",start=now, end=end)
    df_prices.plot()
    df_prices.to_csv(data_path/"price_data_2026se2.csv")


## Starting from SAM processed code

In [ ]:
df_pmax = pd.read_csv(data_path/"sam_wind_farm_production_2024.csv")
df_pmax['Time stamp'] = pd.to_datetime("2024 " + df_pmax['Time stamp'],format="%Y %b %d, %I:%M %p")
df_pmax['Time stamp'] = df_pmax['Time stamp'].dt.tz_localize('+1000')
# df_pmax['Time stamp'] = df_pmax['Time stamp'].dt.tz_convert('Australia/Melbourne')
df_pmax.set_index('Time stamp',inplace=True)
df_pmax = df_pmax[(df_pmax.index<=end) & (df_pmax.index>=now)]
df_pmax_5min = df_pmax.resample("5min").interpolate("linear") # upsample wind to five minutes

In [ ]:
df_prices2 = pd.read_csv(data_path/'price_data_2026se2.csv')

In [ ]:
fig,ax1 = plt.subplots()
ax2 = ax1.twinx()

ax1.plot(df_pmax_5min.index,df_pmax_5min["System power generated | (kW)"], 
         color='tab:red')
ax1.set_ylabel('Wind speed (m/s) @100M', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2.plot( df_prices.index, df_prices['RRP (settlement price, $/MWh)'], 
         color='tab:green')
ax2.set_ylabel('Price ($/MWh)', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

ax1.tick_params(axis='x', rotation=30)

## Set up optimization model

In [ ]:
s0,s_max = 25e3, 50e3 #kWh
Cs,Cg = 0,0 # no costs for now. Ignoring ramp cost in favor of a constraint
P_dis_max, P_chg_max,P_buy_max = 100e3,100e3,0 # kW 
Δt = 5.0/60.0 # hours
P_ramp_max = 0.2*P_dis_max/Δt # kW/hr
P_max = df_pmax_5min['System power generated | (kW)'].values
prices = df_prices["RRP (settlement price, $/MWh)"].values/1e3 # $/kWh

N = prices.shape[0]
model = gp.Model("Wind with battery dispatching")
Ps = model.addVars(range(N),vtype=gp.GRB.CONTINUOUS,lb=0,
                   ub=P_dis_max,name='Power sold')
Pg = model.addVars(range(N),vtype=gp.GRB.CONTINUOUS,lb=0,
                   ub=P_chg_max,name='Power generated')
Pb = model.addVars(range(N),vtype=gp.GRB.CONTINUOUS,lb=0,
                   ub=P_buy_max,name='Power generated')

S = model.addVars(range(N+1),vtype=gp.GRB.CONTINUOUS,lb=0,ub=s_max,name='Stored energy')

# setting the initials to zero since they are in the past
model.addConstr(S[0]==s0)

# storage constraints
model.addConstrs((S[ii+1] == S[ii] - Ps[ii]*Δt + Pg[ii]*Δt + Pb[ii]*Δt for ii in range(N)))

# power capacity constraints
model.addConstrs((Pg[ii]<=P_max[ii] for ii in range(1,N)))
model.addConstrs((Pg[ii]+Pb[ii]<=P_chg_max for ii in range(1,N)))

# ramping constraint (linearized absolute value)
model.addConstrs(((Ps[ii]-Pb[ii]) - (Ps[ii-1] - Pb[ii-1]) <= P_ramp_max*Δt 
                  for ii in range(1,N))) 
model.addConstrs(((Ps[ii-1]-Pb[ii-1]) - (Ps[ii] - Pb[ii]) <= P_ramp_max*Δt for ii in range(1,N)))

# objective (revenue for now)
model.setObjective(Δt*gp.quicksum((  prices[ii]*(Ps[ii]-Pb[ii]) - Cg*Pg[ii] - Cs*Ps[ii] 
                                   for ii in range(1,N))),sense=gp.GRB.MAXIMIZE)


In [ ]:
model.optimize()

In [ ]:
power_sold = []
power_generated = []
power_bought = []
storage = []
revenue = 0.0
for ii in range(N):
    power_sold += [Ps[ii].X]
    power_generated += [Pg[ii].X]
    power_bought += [Pb[ii].X]
    storage += [S[ii].X]
    revenue += prices[ii]*(Ps[ii].X-Pb[ii].X)*Δt

storage += [S[N].X]

power_generated = np.array(power_generated)
power_bought = np.array(power_bought)
power_sold = np.array(power_sold)
storage = np.array(storage)
ramp_rate = np.diff(power_sold-power_bought,prepend=np.nan)/Δt

In [ ]:
fig,ax = plt.subplots(nrows=4,figsize=(7,8))
time = df_prices.index
time_storage = list(time) # storage time is one longer
time_storage += [time_storage[-1]+pd.Timedelta(Δt,'h')]

ax[0].plot(time,(power_sold-power_bought)/1e3,label='Net export')
ax[0].plot(time,power_generated/1e3,label='Generated')
# ax.plot(time,P_max-power_generated/1e3,label='Curtailed')
ax[0].set_ylabel('Power (MW)')
ax[0].set_xticklabels([]) 
ax[0].legend(loc='best')

ax[1].plot(time_storage,storage/1e3,label='Storage',color='green')
ax[1].axhline(s_max/1000,ls='--',color='red',label='Limits')
ax[1].axhline(0,ls='--',color='red')
ax[1].set_xticklabels([]) 
ax[1].set_ylabel("Stored energy (MWh)")
ax[1].legend(loc='best')

ax[2].plot(time,prices*1e3,label='Prices')
ax[2].set_xticklabels([]) 
ax[2].set_ylabel("Prices ($/MWh)")

ax[3].plot(time,ramp_rate/1000.0/60.0,label='Ramp rate')
ax[3].set_ylabel("Ramp rate (Net MW sold/min)")
ax[3].axhline(P_ramp_max/1000/60.0,ls='--',color='red',label='Limits')
ax[3].axhline(-P_ramp_max/1000/60.0,ls='--',color='red')
ax[3].legend(loc='best')

fig.tight_layout()
ax[0].set_title(f'Objective: {model.objVal:.3e} AUD, Revenue: {revenue:.3e} AUD')

